In [1]:
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_platform_name", "cpu")
from jax import vmap, jit
import matplotlib.pyplot as plt
import joblib
from qdots_qll.models import game
from qdots_qll import all_funcs
import seaborn as sns
import pandas as pd
import scipy
from functools import reduce
import os
import re


import matplotlib.font_manager as font_manager

import equinox as eqx

# from matplotlib import rcParams
from scipy.stats import binned_statistic


font = {"family": "Inter"}  # , 'weight': 'normal', 'size': 12}

# Aplica la fuente definida a Matplotlib
plt.rc("font", **font)

sns.set_palette("colorblind")

In [2]:
from scipy.stats import binned_statistic


def compute_mean_one_run(cum_times_i, cov_arr_i, limites_bins):
    indices_bins = np.digitize(cum_times_i, limites_bins)
    means = np.array(
        [
            np.nanmean(cov_arr_i[indices_bins == i], axis=0)
            for i in range(1, len(limites_bins))
        ]
    )

    std_devs = np.array(
        [
            np.nanstd(cov_arr_i[indices_bins == i], axis=0)
            for i in range(1, len(limites_bins))
        ]
    )
    return means, std_devs


names_true = [
    "$\\gamma ( + \\eta)$",
    "$S ( - \\eta)$",
    "$S ( +\\eta)$",
]
names_hat = [
    "$\\hat{\\gamma} ( + \\eta)$",
    "$\\hat{S} ( - \\eta)$",
    "$\\hat{S} ( +\\eta)$",
]


def get_binned_results_from_runs(
    times_array,
    cov_array,
    bin_step,
):
    dims = cov_array[:, 1:].shape[-2:]
    cum_times = np.array(times_array[:, 1:]).cumsum(axis=1)
    cum_times_flatten = cum_times.flatten()
    cov_flatten = np.array(cov_array[:, 1:]).reshape(-1, *dims)
    bins = np.arange(
        cum_times_flatten.min(), cum_times_flatten.max() + 1, bin_step
    )

    mean_list = []
    std_list = []

    for i in range(dims[0]):
        mean_row_list = []
        std_row_list = []

        for j in range(dims[1]):

            mean_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="mean",
                    bins=bins,
                )[0]
            )
            std_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="std",
                    bins=bins,
                )[0]
            )
        mean_list.append(mean_row_list)
        std_list.append(std_row_list)
    mean_list = np.array(mean_list).transpose(2, 1, 0)
    std_list = np.array(std_list).transpose(2, 1, 0)

    cum_times_binned, _, _ = binned_statistic(
        cum_times_flatten, cum_times_flatten, statistic="mean", bins=bins
    )
    return cum_times_binned, mean_list, std_list

In [3]:
filenames = sorted(os.listdir("../results_cluster/one_qdot/"))
filenames = ["../results_cluster/one_qdot/" + i for i in filenames]
job_filenames = list(filter(re.compile(".*job").match, filenames))
log_filenames = list(filter(re.compile(".*log").match, filenames))

In [4]:
r1 = joblib.load(
    "../results_cluster/one_qdot/run_2024-03-28_16:43:31_results.job"
)

In [5]:
runs = [joblib.load(i) for i in job_filenames]

In [6]:
log_filenames

In [7]:
legend_array = [
    "Random",
    "Max det(FIM)",
    "Max trace(FIM)",
    "Random",
    "Max det(FIM)",
    "Max trace(FIM)",
]

In [8]:
runs

In [9]:
len(runs)

In [10]:
i = 0
plt.hist(np.array(runs[i].times_array.flatten()), bins=100)
plt.show()

In [11]:
plt.plot(np.median(runs[3].cov_array, axis=0)[:, 0, 0], label=legend_array[3])

plt.plot(np.median(runs[4].cov_array, axis=0)[:, 0, 0], label=legend_array[4])
plt.plot(np.median(runs[5].cov_array, axis=0)[:, 0, 0], label=legend_array[5])

plt.loglog()
plt.legend()
plt.show()

In [12]:
list_of_results = []

for i in range(len(runs)):
    cum_times_binned, mean_list, std_list = get_binned_results_from_runs(
        runs[i].times_array, runs[i].cov_array, 200
    )

    cov_dict = {"xtimes": cum_times_binned, "mean": mean_list, "std": std_list}
    list_of_results.append(cov_dict)

In [19]:
cum_times_binned

In [25]:
# i = 0

# cum_times_binned, mean_list, std_list = get_binned_results_from_runs(
#     runs[i].times_array, runs[i].cov_array, 200
# )


# cov_dict = {"xtimes": cum_times_binned, "mean": mean_list, "std": std_list}

# fig, axs = plt.subplots(3, 1, figsize=(5, 10))

# for i, ax in enumerate(axs.flat):
#     ax.plot(cum_times_binned, mean_list[:, i, i], "--", ms=0.7)

#     ax.fill_between(
#         cum_times_binned,
#         mean_list[:, i, i] + std_list[:, i, i],
#         mean_list[:, i, i] - std_list[:, i, i],
#         alpha=0.2,
#     )
#     ax.set_title(f"Variance {names_true[i]} ")
#     ax.set_xlabel(f"Total experimental time (ps)")

#     # ax.plot(times, fim_times[:, i, i], label="F")
#     # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
#     # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
#     # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
#     ax.loglog()
#     # ax.legend()
# plt.tight_layout()
# plt.show()

In [19]:
# i = 6
fig, axs = plt.subplots(3, 1, figsize=(5, 10), dpi=200)

for j in range(3):
    for i, ax in enumerate(axs.flat):
        ax.plot(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i],
            "--",
            ms=0.7,
            label=legend_array[j],
        )

        ax.fill_between(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i]
            + list_of_results[j]["std"][:, i, i],
            list_of_results[j]["mean"][:, i, i]
            - list_of_results[j]["std"][:, i, i],
            alpha=0.2,
        )
        ax.loglog()
        ax.set_title(f"Variance {names_true[i]} ")
        ax.set_xlabel(f"Total experimental time (ps)")

        # ax.plot(times, fim_times[:, i, i], label="F")
        # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
        # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
        # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
        ax.legend()
plt.tight_layout()
plt.show()

In [20]:
# i = 6
fig, axs = plt.subplots(3, 1, figsize=(5, 10), dpi=200)

for j in range(3, 6, 1):
    for i, ax in enumerate(axs.flat):
        ax.plot(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i],
            "--",
            ms=0.7,
            label=legend_array[j],
        )

        ax.fill_between(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i]
            + list_of_results[j]["std"][:, i, i],
            list_of_results[j]["mean"][:, i, i]
            - list_of_results[j]["std"][:, i, i],
            alpha=0.2,
        )
        ax.loglog()
        ax.set_title(f"Variance {names_true[i]} ")
        ax.set_xlabel(f"Total experimental time (ps)")

        # ax.plot(times, fim_times[:, i, i], label="F")
        # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
        # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
        # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
        ax.legend()
plt.tight_layout()
plt.show()

In [33]:
import seaborn as sns

In [22]:
fig, axs = plt.subplots(3, 1, dpi=200)  # , figsize=(5, 10))

for i, ax in enumerate(axs.flat):
    ax.hist(
        np.array(runs[i].times_array).flatten(),
        label=legend_array[i],
        bins=100,
    )
    # sns.histplot(np.array(runs[i].times_array).flatten(), bins=100,  kde=False, ax=ax)
    ax.set_xlabel("Experimental time (ps)")
    ax.legend()
plt.suptitle("Chosen times")
plt.tight_layout()
plt.show()

In [23]:
fig, axs = plt.subplots(3, 1, dpi=200)  # , figsize=(5, 10))

for i, ax in enumerate(axs.flat):
    i = i + 3
    ax.hist(
        np.array(runs[i].times_array).flatten(),
        label=legend_array[i],
        bins=100,
    )
    # sns.histplot(np.array(runs[i].times_array).flatten(), bins=100,  kde=False, ax=ax)
    ax.set_xlabel("Experimental time (ps)")
    ax.legend()
plt.suptitle("Chosen times")
plt.tight_layout()
plt.show()

In [24]:
times_boxplot_dict = {}
for i in range(3):
    times_boxplot_dict[legend_array[i]] = np.cumsum(
        runs[i].times_array, axis=1
    )[:, -1]

plt.boxplot(times_boxplot_dict.values(), labels=times_boxplot_dict.keys())
times_boxplot_dict = None
plt.title("Cumulative total experimental time (ps). 20 runs")
plt.tight_layout()
plt.show()

In [25]:
times_boxplot_dict = {}
for i in range(3):
    i = i + 3
    times_boxplot_dict[legend_array[i]] = np.cumsum(
        runs[i].times_array, axis=1
    )[:, -1]

plt.boxplot(times_boxplot_dict.values(), labels=times_boxplot_dict.keys())
times_boxplot_dict = None
plt.title("Cumulative total experimental time (ps). 20 runs")
plt.tight_layout()
plt.show()

# Study of scaling

In [26]:
from qdots_qll.models.models_scratch_for_drafting import (
    SingleQDot3Params,
)
from qdots_qll.utils.povms import sigmas_povm
import qutip as qt

In [27]:
ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()
model = SingleQDot3Params(POVM_array=jnp.array(sigmas_povm))

true_pars = jnp.array([0.35833, 0.053851, -0.333695])

In [28]:
m = model

times = jnp.linspace(0, 40, 500)

fim_times = jax.vmap(lambda t: m.fim(true_pars, t, ground_state_qdot))(times)

norm_times = jax.vmap(lambda a: jnp.linalg.norm(a))(fim_times[:, 0:, 0:])
det_times = jax.vmap(lambda a: jnp.linalg.det(a))(fim_times[:, 0:, 0:])
trace_times = jax.vmap(lambda a: jnp.trace(a))(fim_times[:, 0:, 0:])
fim_inv_times = jax.vmap(lambda a: jnp.linalg.inv(a))(fim_times[:, 0:, 0:])

In [29]:
fig, ax1 = plt.subplots(dpi=200)

color = "tab:red"
ax1.plot(times, det_times, "-.", color=color)
ax1.set_ylabel("Det(FIM)", color=color)
ax1.legend()

ax2 = ax1.twinx()
color = "tab:blue"

ax2.plot(times, trace_times, "-", color=color)
ax2.set_ylabel("Trace(FIM)", color=color)
ax2.legend()

# plt.legend()
plt.tight_layout()
plt.show()

In [30]:
fig, axs = plt.subplots(3, 1, figsize=(5, 10), dpi=200)

for i, ax in enumerate(axs.flatten()):
    for j in range(3, 6, 1):
        ax2 = ax.twinx()
        ax3 = ax.twinx()

        ax.plot(
            times,
            fim_times[:, i, i],
        )
        ax.set_title(f"{names_true[i]}")

        ax2.plot(times, det_times, "-.", color=color, label="Det FIM")
        ax2.legend()
        ax2.yaxis.set_visible(False)

        # ax3.hist(np.array(runs[j].times_array.flatten()), bins=100, alpha=0.3)
        # ax3.yaxis.set_visible(False)

fig.suptitle("FIM and det(FIM)")
plt.tight_layout()
plt.show()

In [32]:
# Factor I_ii integral of the prior. We result is:
#
bnds = np.array([[0.01, 0.9], [0.001, 0.22], [-0.01, -0.9]])
sigmasprior = 5
covs_prior = jnp.diagflat(jnp.std(bnds, axis=1) / sigmasprior) ** 2
mus_prior = jnp.mean(bnds, axis=1)

In [33]:
print(f"mus prior: {mus_prior}")
print(f"covs prior: {np.diag(covs_prior)}")

In [34]:
# Checking standard deviations.
from scipy.stats import norm

rv = norm(loc=mus_prior[0], scale=np.sqrt(covs_prior[0, 0]) * 1)
x = np.linspace(bnds[0, 0], bnds[0, 1], 100)
y = rv.pdf(x)

plt.plot(x, y)

In [35]:
# I = 1/(product of all sigmas) (diag (1/ sigma**2))

# I = (np.sqrt(np.diag(1/(covs_prior))/np.linalg.det(np.sqrt(covs_prior))))
# I = np.diag(I)

I = np.diag(np.diag(1 / covs_prior))
print(I)

# Computation of the expected fisher information

We will use the averaged chosen time to use it in the computation of the FIM.


## Random time

In [36]:
avgtime_random = runs[3].times_array.flatten().mean()
print(avgtime_random)

# Det time

In [37]:
avgtime_det = runs[4].times_array.flatten().mean()
print(avgtime_det)

## Trace time

In [38]:
runs[5].times_array.mean(axis=1)

In [39]:
avgtime_trace = runs[5].times_array.flatten().mean()
print(avgtime_trace)

Now we compute the expected fisher information using the last particle positions (we cannot keep track of everything)

## Expected fisher information

In [40]:
exp_fim_runs_means_list = []

for i in range(0, 6, 1):
    run = runs[i]
    mean_time = jnp.mean(run.times_array, axis=1)
    aux_fim_array = jax.vmap(
        lambda particle_array, t: jax.vmap(
            lambda par, t: m.fim(par, t, ground_state_qdot), in_axes=(0, None)
        )(particle_array, t),
        in_axes=(0, 0),
    )(run.particles_locations, mean_time)

    # print(aux_fim_array.shape)

    exp_fim_runs = jax.vmap(
        lambda weights, arr: jnp.einsum("i, ijk -> jk", weights, arr),
        in_axes=(0, 0),
    )(run.weights, aux_fim_array)
    exp_fim_runs_means = np.mean(exp_fim_runs, axis=0)
    exp_fim_runs_means_list.append(exp_fim_runs_means)

print(np.linalg.det(exp_fim_runs_means))


inv_exp_fim_runs_means_list = []

for i in range(0, 6, 1):
    run = runs[i]
    mean_time = jnp.mean(run.times_array, axis=1)
    aux_inv_fim_array = jax.vmap(
        lambda particle_array, t: jax.vmap(
            lambda par, t: jnp.linalg.inv(m.fim(par, t, ground_state_qdot)),
            in_axes=(0, None),
        )(particle_array, t),
        in_axes=(0, 0),
    )(run.particles_locations, mean_time)

    # print(aux_fim_array.shape)

    inv_exp_fim_runs = jax.vmap(
        lambda weights, arr: jnp.einsum("i, ijk -> jk", weights, arr),
        in_axes=(0, 0),
    )(run.weights, aux_inv_fim_array)
    inv_exp_fim_runs_means = np.mean(inv_exp_fim_runs, axis=0)
    inv_exp_fim_runs_means_list.append(inv_exp_fim_runs_means)

In [41]:
i = 3
print(f"Determinant of EFIM: {np.linalg.det(exp_fim_runs_means_list[i])}")
print(f"1/ E FIM:\n {(1/exp_fim_runs_means_list[i])}")
print(f"E (FIM^-1):\n {(inv_exp_fim_runs_means_list[i])}")

In [42]:
i = 4
print(f"Determinant of EFIM: {np.linalg.det(exp_fim_runs_means_list[i])}")
print(f"1/ E FIM:\n {(1/exp_fim_runs_means_list[i])}")
print(f"E (FIM^-1):\n {(inv_exp_fim_runs_means_list[i])}")

In [43]:
i = 5
print(f"Determinant of EFIM: {np.linalg.det(exp_fim_runs_means_list[i])}")
print(f"1/ E FIM:\n {(1/exp_fim_runs_means_list[i])}")
print(f"E (FIM^-1):\n {(inv_exp_fim_runs_means_list[i])}")

## "True" inverse of FIM

We choose the time that maximizes the determinant and then use the real parameters

In [44]:
best_t = runs[4].times_array.mean()
true_inv_FIM = jnp.linalg.inv(m.fim(true_pars, best_t, ground_state_qdot))
true_FIM = m.fim(true_pars, best_t, ground_state_qdot)

In [45]:
true_inv_FIM

In [46]:
best_t

In [47]:
len(exp_fim_runs_means_list)

In [48]:
# let's stick to det (i=4) s.t. 1/EF is approximate to F^-1

In [49]:
I_inv = np.linalg.inv(I)

In [50]:
i = 4
exp_fim_runs_means_list[i]

In [51]:
inv_exp_fim_runs_means_list[i]

In [52]:
print(np.linalg.inv(exp_fim_runs_means_list[i]))

In [53]:
def scaling(T, EF, I):
    return 1 / T * np.linalg.inv(EF + I / T)
    # return 1 / T * 1/(EF + I / T)

In [54]:
# # i = 6
# fig, axs = plt.subplots(3, 1, figsize=(5, 10))

# for j in range(3, 6, 1):
#     for i, ax in enumerate(axs.flat):
#         ax.plot(
#             list_of_results[j]["xtimes"],
#             list_of_results[j]["mean"][:, i, i],
#             "--",
#             ms=0.7,
#             label=legend_array[j],
#         )

#         ax.fill_between(
#             list_of_results[j]["xtimes"],
#             list_of_results[j]["mean"][:, i, i]
#             + list_of_results[j]["std"][:, i, i],
#             list_of_results[j]["mean"][:, i, i]
#             - list_of_results[j]["std"][:, i, i],
#             alpha=0.2,
#         )
#         ax.loglog()
#         ax.set_title(f"Variance {names_true[i]} ")
#         ax.set_xlabel(f"Total experimental time (ps)")

#         # ax.plot(times, fim_times[:, i, i], label="F")
#         # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
#         # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
#         # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
#         ax.legend()
# plt.tight_layout()
# plt.show()

In [55]:
# j = 5


# cov_scaling = np.array(
#     [
#         scaling(t_i, exp_fim_runs_means_list[j], (I))
#         for t_i in list_of_results[j]["xtimes"]
#     ]
# )
# i = 2
# plt.plot(list_of_results[j]["xtimes"], cov_scaling[:, i, i])

# plt.plot(
#     list_of_results[j]["xtimes"],
#     list_of_results[j]["mean"][:, i, i],
#     "--",
#     ms=0.7,
#     label=legend_array[j],
# )

# plt.loglog()

In [56]:
cov_scaling_list = []

for j in range(6):
    # j = j + 3
    cov_scaling_list.append(
        np.array(
            [
                scaling(t_i, exp_fim_runs_means_list[j], (I))
                for t_i in list_of_results[j]["xtimes"]
            ]
        )
    )

best_scaling_list = []

for j in range(6):
    # j = j + 3
    best_scaling_list.append(
        np.array(
            [
                scaling(t_i, true_FIM, (I))
                for t_i in list_of_results[j]["xtimes"]
            ]
        )
    )

In [59]:
# i = 6
fig, axs = plt.subplots(3, 1, figsize=(5, 10), dpi=200)

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]

for j in range(3, 6, 1):
    for i, ax in enumerate(axs.flat):
        cov_scaling = cov_scaling_list[j]
        best_scaling = best_scaling_list[j]

        ax.plot(
            list_of_results[j]["xtimes"],
            cov_scaling[:, i, i],
            "-.",
            label=f"Scaling {legend_array[j]}",
            color=colors[j - 3],
        )

        ax.plot(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i],
            "--",
            ms=0.7,
            label=legend_array[j],
            color=colors[j - 3],
        )

        ax.fill_between(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i]
            + list_of_results[j]["std"][:, i, i],
            list_of_results[j]["mean"][:, i, i]
            - list_of_results[j]["std"][:, i, i],
            alpha=0.2,
            color=colors[j - 3],
        )
        ax.plot(
            list_of_results[j]["xtimes"],
            best_scaling[:, i, i],
            # label=f"Best Scaling",
            color="black",
            alpha=0.2,
        )
        ax.loglog()
        ax.set_title(f"Variance {names_true[i]} ")
        ax.set_xlabel(f"Total experimental time (ps)")
        ax.set_ylim(
            1e-6,
            0.01,
        )

        # ax.x

        # ax.plot(times, fim_times[:, i, i], label="F")
        # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
        # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
        # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
    ax.legend()
plt.tight_layout()
plt.show()

In [60]:
fig, axs = plt.subplots(3, 1, figsize=(5, 10), dpi=200)

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]
for j in range(3, 6, 1):
    for i, ax in enumerate(axs.flat):
        cov_scaling = cov_scaling_list[j]

        ax.plot(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i],
            "--",
            ms=0.7,
            label=legend_array[j],
            color=colors[j - 3],
        )

        ax.fill_between(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i]
            + list_of_results[j]["std"][:, i, i],
            list_of_results[j]["mean"][:, i, i]
            - list_of_results[j]["std"][:, i, i],
            alpha=0.1,
            color=colors[j - 3],
        )
        ax.plot(
            list_of_results[j]["xtimes"],
            cov_scaling[:, i, i],
            "-",
            alpha=0.6,
            label=f"Scaling {legend_array[j]}",
            color=colors[j - 3],
        )
        ax.loglog()
        ax.set_title(f"Variance {names_true[i]} ")
        ax.set_xlabel(f"Total experimental time (ps)")

        ax.set_xlim(10000)
        ax.set_ylim(
            0.000001,
            0.01,
        )

        # ax.plot(times, fim_times[:, i, i], label="F")
        # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
        # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
        # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
ax.legend()
plt.tight_layout()
plt.show()

## Comparing E(FIM(X)) vs FIM(E(X))

The plots we have made are with E(FIM), let's compute the same with FIM(E(x))

In [61]:
true_pars

In [62]:
runs[4].estimates_array[:, -1].mean(axis=0)

In [63]:
fim_exp_pars_means_list = []
inv_fim_exp_pars_means_list = []
for i in range(0, 6, 1):

    run = runs[i]
    mean_time = jnp.mean(run.times_array, axis=1)

    e_particle = run.estimates_array[:, -1]

    fim_expparticle_array = jax.vmap(
        lambda par, t: m.fim(par, t, ground_state_qdot), in_axes=(0, 0)
    )(e_particle, mean_time)

    inv_fim_expparticle_array = jax.vmap(
        lambda par, t: jnp.linalg.inv(m.fim(par, t, ground_state_qdot)),
        in_axes=(0, 0),
    )(e_particle, mean_time)

    fim_exp_pars_means = np.mean(fim_expparticle_array, axis=0)
    fim_exp_pars_means_list.append(fim_exp_pars_means)

    inv_fim_exp_pars_means = np.mean(inv_fim_expparticle_array, axis=0)
    inv_fim_exp_pars_means_list.append(inv_fim_exp_pars_means)

In [64]:
fim_exp_cov_scaling_list = []

for j in range(6):
    # j = j + 3
    fim_exp_cov_scaling_list.append(
        np.array(
            [
                scaling(t_i, fim_exp_pars_means_list[j], (I))
                for t_i in list_of_results[j]["xtimes"]
            ]
        )
    )

In [119]:
fig, axs = plt.subplots(3, 1, figsize=(5, 10), dpi=200)

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]
for j in range(3, 6, 1):
    for i, ax in enumerate(axs.flat):
        cov_scaling = fim_exp_cov_scaling_list[j]

        ax.plot(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i],
            "--",
            ms=0.7,
            label=legend_array[j],
            color=colors[j - 3],
        )

        ax.fill_between(
            list_of_results[j]["xtimes"],
            list_of_results[j]["mean"][:, i, i]
            + list_of_results[j]["std"][:, i, i],
            list_of_results[j]["mean"][:, i, i]
            - list_of_results[j]["std"][:, i, i],
            alpha=0.1,
            color=colors[j - 3],
        )
        ax.plot(
            list_of_results[j]["xtimes"],
            cov_scaling[:, i, i],
            "-",
            alpha=0.6,
            label=f"Scaling {legend_array[j]}",
            color=colors[j - 3],
        )
        ax.loglog()
        ax.set_title(f"Variance {names_true[i]} ")
        ax.set_xlabel(f"Total experimental time (ps)")

        ax.set_xlim(10000)
        ax.set_ylim(
            0.000001,
            0.01,
        )

        # ax.plot(times, fim_times[:, i, i], label="F")
        # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
        # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
        # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
ax.legend()
plt.tight_layout()
plt.show()

In [202]:
list_of_results[j]["xtimes"].shape

In [367]:
A = np.random.rand(3, 3)
B = np.random.rand(3, 3)

In [373]:
np.linalg.matrix_rank(B)

In [371]:
np.linalg.inv(A + B)

In [372]:
np.linalg.inv(A) - 1 / (1 + np.trace(np.linalg.inv(A) @ B)) * np.linalg.inv(
    A
) @ B @ np.linalg.inv(A)

## Using generalized cov

In [74]:
list_of_results[0]["mean"].shape

jax.vmap(lambda mean_array: jnp.linalg.det(mean_array))(list_of_results[0]["mean"]).shape

plt.plot(list_of_results[0]["xtimes"],jax.vmap(lambda mean_array: jnp.linalg.det(mean_array))(list_of_results[0]["mean"]) )
plt.loglog()
plt.show()


In [87]:
def det_scaling(T, EF, I):
    return  np.linalg.det(1 / T *np.linalg.inv(EF + I / T))

In [95]:
det_cov_scaling_list = []

for j in range(6):
    # j = j + 3
    det_cov_scaling_list.append(
        np.array(
            [
                det_scaling(t_i, exp_fim_runs_means_list[j], (I))
                for t_i in list_of_results[j]["xtimes"]
            ]
        )
    )


In [96]:
# i = 6
fig, axs = plt.subplots(1, 1, figsize=(5, 10), dpi=200)

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]

for j in range(3, 6, 1):

    # cov_scaling = cov_scaling_list[j]
    # best_scaling = best_scaling_list[j]

    axs.plot(list_of_results[j]["xtimes"],jax.vmap(lambda mean_array: jnp.linalg.det(mean_array))(list_of_results[j]["mean"]),label=legend_array[j],color=colors[j - 3], )
    
    axs.plot(list_of_results[j]["xtimes"], det_cov_scaling_list[j], "-.", color=colors[j - 3],)

    # ax.plot(
    #     list_of_results[j]["xtimes"],
    #     cov_scaling[:, i, i],
    #     "-.",
    #     label=f"Scaling {legend_array[j]}",
    #     color=colors[j - 3],
    # )
    # 
    # ax.plot(
    #     list_of_results[j]["xtimes"],
    #     list_of_results[j]["mean"][:, i, i],
    #     "--",
    #     ms=0.7,
    #     label=legend_array[j],
    #     color=colors[j - 3],
    # )

    # ax.fill_between(
    #     list_of_results[j]["xtimes"],
    #     list_of_results[j]["mean"][:, i, i]
    #     + list_of_results[j]["std"][:, i, i],
    #     list_of_results[j]["mean"][:, i, i]
    #     - list_of_results[j]["std"][:, i, i],
    #     alpha=0.2,
    #     color=colors[j - 3],
    # )
    # ax.plot(
    #     list_of_results[j]["xtimes"],
    #     best_scaling[:, i, i],
    #     # label=f"Best Scaling",
    #     color="black",
    #     alpha=0.2,
    # )
    axs.loglog()
    axs.set_title(f"Variance {names_true[i]} ")
    axs.set_xlabel(f"Total experimental time (ps)")
    # axs.set_ylim(
    #     1e-6,
    #     0.01,
    # )

    # ax.x

    # ax.plot(times, fim_times[:, i, i], label="F")
    # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
    # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
    # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
axs.legend()
plt.tight_layout()
plt.show()

# Computing the fisher information with the expected estimates at each step

In [97]:
run

In [112]:
cum_times = jnp.mean(run.times_array.cumsum(axis=1), axis=0)
estimates = jnp.mean(run.estimates_array[:], axis=0)



fim_times = jax.vmap(
    lambda particle_array, t: 
       m.fim(particle_array, t, ground_state_qdot),
    in_axes=(0, 0),
)(estimates, cum_times)



In [113]:
fim_times.shape

In [117]:
fim_exp_per_step_list = []
# inv_fim_exp_per_step_list = []
for i in range(0, 6, 1):

    run = runs[i]
    run
    # cum_times = jnp.mean(run.times_array.cumsum(axis=1), axis=0)
    estimates = jnp.mean(run.estimates_array[:], axis=0)
    
    fim_times = jax.vmap(
        lambda particle_array, t:
        m.fim(particle_array, t, ground_state_qdot),
        in_axes=(0, 0),
    )(estimates, jnp.mean(run.times_array, axis=0))

    # fim_exp_pars_means = np.mean(fim_expparticle_array, axis=0)
    fim_exp_per_step_list.append(fim_times)

    # inv_fim_exp_pars_means = np.mean(inv_fim_expparticle_array, axis=0)
    # inv_fim_exp_per_step_list.append(inv_fim_exp_pars_means)

In [131]:
i = 5

In [132]:
fim_exp_pars_means_list[i][-1]

In [133]:
fim_exp_per_step_list[i][-1]

In [128]:
i = 4

cumtimes = cum_times = jnp.mean(run.times_array.cumsum(axis=1), axis=0)
fim_exp_times = fim_exp_per_step_list[i]

jax.vmap((lambda T, EF, I: 1/(T**3)* jnp.linalg.det((EF + I / T)) ), in_axes=(0, 0, None))(cumtimes, fim_exp_times, I)





In [134]:
det_cov_scaling_step_list = []

for j in range(6):
    # run = runs[j]
    cumtimes =  jnp.mean(runs[j].times_array.cumsum(axis=1), axis=0)
    fim_exp_times = fim_exp_per_step_list[j]
    
    

# j = j + 3
    det_cov_scaling_step_list.append(
        np.array(jax.vmap((lambda T, EF, I: 1/(T**3)* jnp.linalg.det((EF + I / T)) ), in_axes=(0, 0, None))(cumtimes, fim_exp_times, I))
    )

In [ ]:
# i = 6
fig, axs = plt.subplots(1, 1, figsize=(5, 10), dpi=200)

colors = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
]

for j in range(3, 6, 1):

    # cov_scaling = cov_scaling_list[j]
    # best_scaling = best_scaling_list[j]

    axs.plot(list_of_results[j]["xtimes"],jax.vmap(lambda mean_array: jnp.linalg.det(mean_array))(list_of_results[j]["mean"]),label=legend_array[j],color=colors[j - 3], )

    axs.plot(list_of_results[j]["xtimes"], det_cov_scaling_step_list[j], "-.", color=colors[j - 3],)

    # ax.plot(
    #     list_of_results[j]["xtimes"],
    #     cov_scaling[:, i, i],
    #     "-.",
    #     label=f"Scaling {legend_array[j]}",
    #     color=colors[j - 3],
    # )
    # 
    # ax.plot(
    #     list_of_results[j]["xtimes"],
    #     list_of_results[j]["mean"][:, i, i],
    #     "--",
    #     ms=0.7,
    #     label=legend_array[j],
    #     color=colors[j - 3],
    # )

    # ax.fill_between(
    #     list_of_results[j]["xtimes"],
    #     list_of_results[j]["mean"][:, i, i]
    #     + list_of_results[j]["std"][:, i, i],
    #     list_of_results[j]["mean"][:, i, i]
    #     - list_of_results[j]["std"][:, i, i],
    #     alpha=0.2,
    #     color=colors[j - 3],
    # )
    # ax.plot(
    #     list_of_results[j]["xtimes"],
    #     best_scaling[:, i, i],
    #     # label=f"Best Scaling",
    #     color="black",
    #     alpha=0.2,
    # )
    axs.loglog()
    axs.set_title(f"Variance {names_true[i]} ")
    axs.set_xlabel(f"Total experimental time (ps)")
    # axs.set_ylim(
    #     1e-6,
    #     0.01,
    # )

    # ax.x

    # ax.plot(times, fim_times[:, i, i], label="F")
    # ax.plot(times, inv_fim_times[:, i, i], label="Inverse")
    # ax.plot(times, 1/fim_times[:, i, i], label="1/F")
    # ax.axvline( jnp.min(inv_fim_times[:, i, i]))
axs.legend()
plt.tight_layout()
plt.show()